# Native RL training status

This notebook monitors the fresh-rollout training run without changing it. It plots completed rounds, PPO loss, transitions, rollout outcomes, and an ETA for the configured 600-round run. Outcome curves are learning proxies, not a substitute for held-out evaluation against fixed opponents.

In [ ]:
from pathlib import Path
import json, re, time
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = Path('/home/justin/p/poke_learning_2')
ROLLOUT_DIR = ROOT / 'data/logs/champions_mb_fresh_rollouts'
LOG_PATH = Path('/tmp/champions_mb_fresh_rl.log')
EXTENSION_LOG_PATH = Path('/tmp/champions_mb_fresh_rl_extension.log')
TOTAL_ROUNDS = 600
EPISODES_PER_ROUND = 32

In [ ]:
def training_rows():
    rows = []
    pattern = re.compile(r"\{'update':\s*(\d+),\s*'loss':\s*([-+0-9.eE]+),\s*'transitions':\s*([-+0-9.eE]+)\}")
    if LOG_PATH.exists():
        text = LOG_PATH.read_text(errors='replace')
        round_number = 0
        for line in text.splitlines():
            match = pattern.search(line)
            if match:
                update, loss, transitions = match.groups()
                rows.append({'round': round_number + 1, 'update': int(update), 'loss': float(loss), 'transitions': float(transitions)})
            completed = re.search(r'completed_round=(\d+)', line)
            if completed:
                round_number = int(completed.group(1))
    return rows

def rollout_rows():
    rows = []
    for path in sorted(ROLLOUT_DIR.glob('round_*.jsonl')):
        episodes, wins, rewards, steps = 0, 0, [], []
        for line in path.read_text(errors='replace').splitlines():
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            episodes += 1
            steps.append(record.get('steps', 0))
            result = record.get('rewards', {})
            reward = result.get('player1', result.get('agent1', 0)) if isinstance(result, dict) else result
            rewards.append(float(reward or 0))
            if float(reward or 0) > 0:
                wins += 1
        if episodes:
            rows.append({'round': int(path.stem.split('_')[-1]), 'episodes': episodes, 'win_rate_proxy': wins / episodes, 'mean_reward': np.mean(rewards), 'mean_steps': np.mean(steps)})
    return rows

def snapshot():
    train, rollouts = training_rows(), rollout_rows()
    completed = max([r['round'] for r in rollouts], default=0)
    remaining = max(TOTAL_ROUNDS - completed, 0)
    if len(rollouts) >= 2:
        elapsed = (max(p.stat().st_mtime for p in ROLLOUT_DIR.glob('round_*.jsonl')) - min(p.stat().st_mtime for p in ROLLOUT_DIR.glob('round_*.jsonl')))
        rate = (len(rollouts) - 1) / max(elapsed, 1)
        eta_hours = remaining / max(rate, 1e-9) / 3600
    else:
        eta_hours = float('nan')
    return train, rollouts, completed, eta_hours

In [ ]:
train, rollouts, completed, eta_hours = snapshot()
print(f'Completed rounds: {completed}/{TOTAL_ROUNDS}')
print(f'Collected games: {sum(r["episodes"] for r in rollouts):,}')
print(f'Estimated time remaining: {eta_hours:.2f} hours')
if train:
    print(f'Latest PPO loss: {train[-1]["loss"]:.6f}; latest transitions: {train[-1]["transitions"]:,.0f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
if train:
    axes[0, 0].plot([r['round'] for r in train], [r['loss'] for r in train], '.', alpha=.35)
    axes[0, 0].set(title='PPO loss by update', xlabel='Round', ylabel='Loss')
    axes[0, 1].plot([r['round'] for r in train], [r['transitions'] for r in train], '.', alpha=.35)
    axes[0, 1].set(title='Transitions per update', xlabel='Round', ylabel='Transitions')
if rollouts:
    x = [r['round'] for r in rollouts]
    axes[1, 0].plot(x, [r['win_rate_proxy'] for r in rollouts], '-o', ms=3)
    axes[1, 0].set(title='Self-play outcome proxy', xlabel='Round', ylabel='Player 1 reward-positive rate')
    axes[1, 1].plot(x, [r['mean_steps'] for r in rollouts], '-o', ms=3, label='Mean steps')
    axes[1, 1].set(title='Battle behavior over time', xlabel='Round', ylabel='Steps')
for ax in axes.flat:
    ax.grid(alpha=.25)
plt.show()

## Live monitoring

Rerun the snapshot and plotting cells periodically. In JupyterLab, enabling an autoreload/refresh extension is optional; the simplest approach is to rerun the cells every few minutes. For actual ability improvement, add held-out checkpoint-vs-baseline results here once the native matchup evaluator is available.